In [17]:
import time

import numpy as np
import pytz

from Data_query.trino_config import *
from visualisation import *

In [13]:
stop_trino()

Trino service stopping triggered.


In [27]:
sleep(60)

In [26]:
big_workers = 1
workers = 0
num_workers = max(workers, big_workers)
ensure_trino_running(worker_desired_count=workers, big_worker_desired_count=big_workers)
# sleep(40)

Trino service is already running.


In [24]:
iceberg_sql(f""" select distinct site_id, ac_capacity_kw, state, min_time, max_time from meta_up23c 
            where is_pv=True and flex_export_detected=False and pf_01 > .98 and min_time < timestamp '2024-01-02'
            limit 2""")

,site_id,ac_capacity_kw,state,min_time,max_time
0,1233585204,5.0,VIC,2024-01-01,2024-06-04 20:55:00
1,158521185,8.0,NSW,2024-01-01,2025-06-06 07:10:00


In [ ]:
# Some sites are excluded because no CS day profile is detected for them. For example, site 1525233041, cs_day is 2024-01-21. But no ts data is detected for that day.

In [28]:
iceberg_exec("drop table if exists single_site_pv_ghi_model")
iceberg_exec(f"""
     create table single_site_pv_ghi_model (
            site_id BIGINT,
            tod_bin TIME,
            a double,
            b double,
            n BIGINT
        )""")

Executed
Executed


In [11]:
sleep(20)

In [29]:
num_parts = 1
time_bin_interval = "5"  # in minutes


def run_func(args):
    t00 = time.time()
    year, month, part = args
    time_filter = f"year = {year}"
    part_filter = f"site_id % {num_parts} = {part}"
    # meta_filters = f"is_pv=True and {split_cons} and flex_export_detected=False and {part_filter}"
    # meta_filters = f"is_pv=True and {split_cons} and flex_export_detected=False and site_id in (250484079)"
    site_filters = f"site_id in (1567203727, 250484079, 1792599725, 696192939)"
    # meta_filters = f"is_pv=True and {split_cons} and flex_export_detected=False and site_id in (1792599725)"
    df = iceberg_exec(f"""
                    insert into single_site_pv_ghi_model
                    with train_val_data AS (
                        SELECT
                            site_id,
                            actual_day,
                            t_stamp,
                            CAST(
                                date_trunc('minute', t_stamp + interval '10' hour)
                                - interval '1' minute * (minute(t_stamp + interval '10' hour) % {time_bin_interval})
                                AS TIME) AS tod_bin,
                            GHI/GHI_cs AS x,
                            GHI_cs,
                            P_kw_norm/ NULLIF(P_kw_norm_cs, 0.0) AS y
                        FROM structured_data
                        WHERE P_kw_norm_cs > 0.2 AND GHI > 50 and P_kw_norm > 0.05 and P_kw_norm <= P_kw_norm_cs
                            and V <= 253 and (P_kw_norm >= 1 or S_norm < 1.001) and {time_filter} and {site_filters}
                    ),
                    train_data as (
                        SELECT t.*
                        FROM train_val_data t JOIN split_days s ON t.site_id = s.site_id AND t.actual_day = s.actual_day
                        WHERE s.day_type = 'train'),
                    model AS (
                        SELECT
                            site_id,
                            tod_bin,
                            (1 - regr_slope(y, x)) AS a,
                            regr_slope(y, x)     AS b,
                            count(*)             AS n
                        FROM train_data
                        GROUP BY site_id, tod_bin
                    )
                    select * from model
                    
    """)

    print(
        f"Completed {time_filter}, part {part}, time taken: {round(time.time() - t00, 2)} seconds"
    )
    return df


tasks = [
    (year, month, part)
    for year in (2024,)
    for month in range(1, 2)
    for part in range(0, num_parts)
]
#   for split_cons in ['system.bucket(postcode, 16) > -1'] ]

try:
    res = trino_parallel(run_func, tasks, num_workers=num_workers)
except Exception as e:
    print(f"Error during data retrieval: {e}")
finally:
    # stop_trino()
    pass
res

Executed
Completed year = 2024, part 0, time taken: 5.52 seconds


In [8]:
365 - 52 * 2

261

In [16]:
iceberg_sql(f"""select * from single_site_pv_ghi_model
            order by abs(a) desc""")

,site_id,tod_bin,a,b,n
0,1567203727,15:40:00,-18.202578,19.202578,9
1,1567203727,15:25:00,-17.920507,18.920507,10
2,1567203727,15:35:00,-16.762831,17.762831,9
3,1792599725,05:50:00,-7.126259,8.126259,11
4,1567203727,10:45:00,-6.246858,7.246858,8
...,...,...,...,...,...
493,250484079,14:30:00,-0.003419,1.003419,229
494,1792599725,09:55:00,0.001487,0.998513,178
495,1792599725,10:05:00,-0.001101,1.001101,165
496,250484079,14:05:00,-0.000272,1.000272,228


In [22]:
num_parts = 1
time_bin_interval = "5"  # in minutes


def run_func(args):
    year, month, split_cons, part = args
    time_filter = f"year = {year}"
    part_filter = f"site_id % {num_parts} = {part}"
    # meta_filters = f"is_pv=True and {split_cons} and flex_export_detected=False and {part_filter}"
    meta_filters = f"is_pv=True and {split_cons} and flex_export_detected=False and site_id in (1792599725)"
    df = iceberg_sql(f"""
                    with data as 
                        (select 
                            site_id, t_stamp,  sum(power*circuit_polarity/S_99)/1000 as P_kw_norm,
                            sum(energy_reactive*circuit_polarity/S_99)/1000*12 as Q_kvar_norm, 
                            avg(voltage) as V
                        from ts join 
                            (select site_id, circuit_id, circuit_polarity, S_99  from meta_up23c where {meta_filters})
                            as m on ts.circuit_id = m.circuit_id
                        where {time_filter} and {part_filter} and is_pv=True and voltage >= 200 and voltage <= 300 and {split_cons}
                        group by site_id, t_stamp
                            ),
                    bom10min as (
                        select 
                            distinct time, b.latitude, b.longitude, surface_global_irradiance as GHI, cloud_type
                        from bom_nci.solar as b
                            join (select distinct site_id, n_lat, n_long from meta_up23c where {meta_filters}) as m 
                            on b.latitude = m.n_lat and b.longitude = m.n_long
                        where {time_filter} 
                    ),
                    bom5min as ((select time as time_5min, latitude, longitude, GHI, cloud_type
                    from bom10min) 
                    union all
                    (select date_add('minute', 5, time) as time_5min, latitude, longitude, GHI, cloud_type
                    FROM bom10min ORDER BY time_5min)),
                    daily_cloud AS (
                        SELECT
                            latitude, longitude,
                            date_trunc('day', time + interval '10' hour)   AS day,
                            date_trunc('month', time + interval '10' hour) AS month,
                            sum(cloud_type) AS cloud_sum, 
                            max(GHI) AS max_GHI
                        FROM bom10min
                        GROUP BY
                            1, 2, 3, 4
                    ),
                    clear_sky AS (
                            SELECT day, latitude, longitude
                            FROM (SELECT day, latitude, longitude, cloud_sum, max_GHI, row_number() OVER (
                                                                    PARTITION BY month, latitude, longitude
                                                                    ORDER BY cloud_sum ASC, day ASC
                                                                    ) AS rn
                                FROM daily_cloud 
                            )
                            WHERE rn < 4 and cloud_sum < 60 and max_GHI > 200
                        ),
                    daily_site_days AS (
                            SELECT 
                                n_lat,
                                n_long,
                                date_trunc('day', t_stamp + interval '10' hour) AS day
                            FROM data d join (select distinct site_id, n_lat, n_long from meta_up23c where {meta_filters}) m
                            on d.site_id = m.site_id
                            group by n_lat, n_long, date_trunc('day', t_stamp + interval '10' hour) 
                        ),
                    nearest_clear_sky_day AS (
                        SELECT
                            dy.n_lat,
                            dy.n_long,
                            dy.day AS actual_day,
                            c.day AS clear_sky_day,
                            row_number() OVER (
                                PARTITION BY dy.n_lat, dy.n_long, dy.day
                                ORDER BY abs(date_diff('day', dy.day, c.day)), date_diff('day', c.day, dy.day)
                            ) AS rn
                        FROM daily_site_days dy JOIN clear_sky c
                            ON dy.n_lat = c.latitude AND dy.n_long = c.longitude
                    ),
                    nearest_clear_sky AS (
                            SELECT
                                n_lat,
                                n_long,
                                actual_day,
                                clear_sky_day
                            FROM nearest_clear_sky_day
                            WHERE rn = 1
                        ),
                    nearest_cs_days AS (
                        SELECT
                            DISTINCT site_id, clear_sky_day AS cs_day 
                        FROM nearest_clear_sky n JOIN (select distinct site_id, n_lat, n_long from meta_up23c where {meta_filters}) m
                            ON n.n_lat = m.n_lat AND n.n_long = m.n_long
                    ),
                    base AS (
                            SELECT
                                d.*,
                                lag(t_stamp) OVER (
                                    PARTITION BY site_id, date_trunc('day', t_stamp + interval '10' hour)
                                    ORDER BY t_stamp
                                ) AS prev_ts
                            FROM data d
                    ),
                    gaps AS (
                        SELECT *,
                            CASE
                                WHEN prev_ts IS NULL THEN 0
                                WHEN t_stamp - prev_ts > interval '30' minute THEN 1
                                ELSE 0
                            END AS gap_start
                        FROM base
                    ),
                    segments AS (
                        SELECT *,
                            sum(gap_start) OVER (
                                PARTITION BY site_id, date_trunc('day', t_stamp + interval '10' hour)
                                ORDER BY t_stamp
                                ROWS UNBOUNDED PRECEDING
                            ) AS segment_id
                        FROM gaps
                    ),
                    nearest_cs_profiles AS (
                        SELECT
                            s.site_id,
                            date_trunc('day', s.t_stamp + interval '10' hour) AS cs_day,
                            (s.t_stamp + interval '10' hour) - date_trunc('day', s.t_stamp + interval '10' hour) AS cs_tod,
                            approx_percentile(P_kw_norm, 0.6) OVER (
                                    PARTITION BY s.site_id, date_trunc('day', s.t_stamp + interval '10' hour), segment_id
                                        ORDER BY t_stamp
                                        ROWS BETWEEN 3 PRECEDING AND 3 FOLLOWING
                                    ) AS P_kw_norm_cs,
                            approx_percentile(GHI, 0.6) OVER (
                            PARTITION BY s.site_id, date_trunc('day', s.t_stamp + interval '10' hour), segment_id
                                ORDER BY t_stamp
                                ROWS BETWEEN 3 PRECEDING AND 3 FOLLOWING
                            ) AS GHI_cs,
                            cloud_type as cloud_type_cs
                        FROM segments s join nearest_cs_days n on s.site_id = n.site_id and 
                            date_trunc('day', s.t_stamp + interval '10' hour) = n.cs_day
                            join (select distinct site_id, n_lat, n_long from meta_up23c where {meta_filters}) m on s.site_id = m.site_id
                            join bom5min b on m.n_lat = b.latitude and m.n_long = b.longitude and b.time_5min = s.t_stamp
                    ),
                    strcutured_data AS (
                        SELECT
                            d.site_id,
                            d.t_stamp,
                            date_trunc('day', d.t_stamp + interval '10' hour) AS actual_day,
                            (d.t_stamp + interval '10' hour) - date_trunc('day', d.t_stamp + interval '10' hour) AS actual_tod,
                            d.V,
                            d.Q_kvar_norm,
                            d.P_kw_norm, 
                            sqrt(pow(d.Q_kvar_norm, 2) + pow(d.P_kw_norm, 2)) AS S_norm,
                            GHI,
                            cloud_type,
                            ncs.cs_day,
                            ncs.cs_tod,
                            ncs.P_kw_norm_cs,
                            ncs.GHI_cs, 
                            ncs.cloud_type_cs
                        FROM data d 
                            join (select distinct site_id, n_lat, n_long from meta_up23c where {meta_filters}) m
                                ON d.site_id = m.site_id 
                            join nearest_cs_profiles ncs 
                                on d.site_id = ncs.site_id and ncs.cs_tod = (d.t_stamp + interval '10' hour) - date_trunc('day', d.t_stamp + interval '10' hour)
                            join nearest_clear_sky n on 
                                n.n_lat = m.n_lat AND n.n_long = m.n_long and n.actual_day = date_trunc('day', d.t_stamp + interval '10' hour)
                                and n.clear_sky_day = ncs.cs_day
                            join bom5min b on m.n_lat = b.latitude and m.n_long = b.longitude and b.time_5min = d.t_stamp
                        Where abs(date_diff('day', ncs.cs_day, n.actual_day)) < 45
                            order by d.site_id, d.t_stamp),
                    train_val_data AS (
                        SELECT
                            site_id,
                            actual_day,
                            cs_day,
                            t_stamp,
                            CAST(
                                date_trunc('minute', t_stamp + interval '10' hour)
                                - interval '1' minute * (minute(t_stamp + interval '10' hour) % {time_bin_interval})
                                AS TIME) AS tod_bin,
                            GHI/NULLIF(GHI_cs, 0.0) AS x,
                            GHI,
                            GHI_cs,
                            P_kw_norm/ NULLIF(P_kw_norm_cs, 0.0) AS y,
                            P_kw_norm,
                            P_kw_norm_cs
                        FROM strcutured_data
                        WHERE P_kw_norm_cs > 0.2 AND GHI > 50 and P_kw_norm > 0.05 and P_kw_norm <= P_kw_norm_cs
                            and V <= 253 and (P_kw_norm >= 1 or S_norm < 1.001)
                    ),
                    train_data as (
                        SELECT t.*
                        FROM train_val_data t JOIN split_days s ON t.site_id = s.site_id AND t.actual_day = s.actual_day
                        WHERE s.day_type = 'train')
                    select * from train_data where t_stamp < timestamp '2024-01-05 00:00:00'
                    
    """)

    print(
        f"Completed {time_filter}, {split_cons.replace('system.bucket(postcode, 16)', 'bucket')}, part {part}"
    )
    return df


tasks = [
    (year, month, split_cons, part)
    for year in (2024,)
    for month in range(1, 2)
    for split_cons in [f"system.bucket(postcode, 16) = {i}" for i in range(16)]
    for part in range(0, num_parts)
]
#   for split_cons in ['system.bucket(postcode, 16) > -1'] ]

try:
    res = trino_parallel(run_func, tasks, num_workers=num_workers)
except Exception as e:
    print(f"Error during data retrieval: {e}")
finally:
    # stop_trino()
    pass
res

Completed year = 2024, bucket = 0, part 0
Completed year = 2024, bucket = 1, part 0
Completed year = 2024, bucket = 2, part 0
Completed year = 2024, bucket = 3, part 0
Completed year = 2024, bucket = 4, part 0
Completed year = 2024, bucket = 5, part 0
Completed year = 2024, bucket = 6, part 0
Completed year = 2024, bucket = 7, part 0
Completed year = 2024, bucket = 8, part 0
Completed year = 2024, bucket = 9, part 0
Completed year = 2024, bucket = 10, part 0
Completed year = 2024, bucket = 11, part 0
Completed year = 2024, bucket = 12, part 0
Completed year = 2024, bucket = 13, part 0
Completed year = 2024, bucket = 14, part 0
Completed year = 2024, bucket = 15, part 0
Combining results from all tasks.


,site_id,actual_day,cs_day,t_stamp,tod_bin,x,GHI,GHI_cs,y,P_kw_norm,P_kw_norm_cs
0,1792599725,2024-01-01,2024-01-21,2024-01-01 01:15:00,11:15:00,0.993662,1064.51,1071.30,0.940794,0.867038,0.921602
1,1792599725,2024-01-01,2024-01-21,2024-01-01 02:20:00,12:20:00,0.605380,658.69,1088.06,0.521317,0.472019,0.905436
2,1792599725,2024-01-03,2024-01-21,2024-01-02 21:50:00,07:50:00,1.051483,578.20,549.89,0.973780,0.665304,0.683217
3,1792599725,2024-01-03,2024-01-21,2024-01-02 22:35:00,08:35:00,0.982330,713.26,726.09,0.980257,0.808117,0.824393
4,1792599725,2024-01-03,2024-01-21,2024-01-03 03:25:00,13:25:00,0.800899,828.13,1034.00,0.366897,0.330235,0.900074
...,...,...,...,...,...,...,...,...,...,...,...
172,1792599725,2024-01-04,2024-01-21,2024-01-04 03:40:00,13:40:00,0.461475,470.46,1019.47,0.855322,0.764191,0.893453
173,1792599725,2024-01-04,2024-01-21,2024-01-04 03:45:00,13:45:00,0.468964,470.46,1003.19,0.384118,0.340096,0.885395
174,1792599725,2024-01-04,2024-01-21,2024-01-04 04:55:00,14:55:00,0.284018,239.68,843.89,0.278267,0.203175,0.730145
175,1792599725,2024-01-05,2024-01-21,2024-01-04 21:35:00,07:35:00,0.980525,502.46,512.44,0.975059,0.612573,0.628242


In [23]:
res.query("actual_day=='2024-01-03'")

,site_id,actual_day,cs_day,t_stamp,tod_bin,x,GHI,GHI_cs,y,P_kw_norm,P_kw_norm_cs
2,1792599725,2024-01-03,2024-01-21,2024-01-02 21:50:00,07:50:00,1.051483,578.20,549.89,0.973780,0.665304,0.683217
3,1792599725,2024-01-03,2024-01-21,2024-01-02 22:35:00,08:35:00,0.982330,713.26,726.09,0.980257,0.808117,0.824393
4,1792599725,2024-01-03,2024-01-21,2024-01-03 03:25:00,13:25:00,0.800899,828.13,1034.00,0.366897,0.330235,0.900074
5,1792599725,2024-01-03,2024-01-21,2024-01-03 03:40:00,13:40:00,0.783682,798.94,1019.47,0.848862,0.758419,0.893453
6,1792599725,2024-01-03,2024-01-21,2024-01-03 04:00:00,14:00:00,0.768798,757.42,985.20,0.307086,0.260322,0.847718
...,...,...,...,...,...,...,...,...,...,...,...
162,1792599725,2024-01-03,2024-01-21,2024-01-03 01:50:00,11:50:00,0.988878,1074.93,1087.02,0.563877,0.519031,0.920468
163,1792599725,2024-01-03,2024-01-21,2024-01-03 02:30:00,12:30:00,0.880542,956.04,1085.74,0.676370,0.614689,0.908805
164,1792599725,2024-01-03,2024-01-21,2024-01-03 04:15:00,14:15:00,0.661891,593.71,896.99,0.322501,0.265061,0.821890
165,1792599725,2024-01-03,2024-01-21,2024-01-03 04:20:00,14:20:00,0.764334,685.60,896.99,0.348573,0.282554,0.810600
